# SOLUCIÓN PROBLEMA 1 - FINAL Probabilidad y Estadística - iMAT-ICAI - 19 de mayo 2023

# Vale el 30% de la nota del examen
---

NO CAMBIES EL NOMBRE AL NOTEBOOK, SIMPLEMENTE ESCRIBE AQUÍ TUS DATOS
### Nombre: RELLENAR
### Apellidos: RELLENAR
---

### ¿Cuánto dura el cable del cargador del teléfono móvil? Una empresa quiere modelar el tiempo hasta el fallo de uno de sus modelos para poder tomar decisiones que les permita obtener un nuevo producto competitivo.

### Para ello dispone de una muestra de tiempos hasta el fallo, medido en meses, recopilados para su modelo de cable de 1.5 metros, el más vendido. 

Responde las preguntas utilizando instrucciones print() para mostrar claramente tu respuesta en la ejecución de la celda, explicando cómo lo resuelves. Utiliza comentarios en tu código para facilitar su lectura.

### Importamos todos los paquetes aquí por comodidad

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seaborn as sns
from scipy import optimize as opt  # para estimación por máxima verosimilitud

from statsmodels.nonparametric.kde import KDEUnivariate

### La muestra disponible se ha conseguido a partir de la fecha de compra y la fecha en la que los usuarios han solicitado la devolución del dinero, para aquellos casos en los que se ha recibido el cable antiguo, confirmado que está deteriorado por el uso y que se corresponde con el que se compró.

In [ ]:
# Carga los datos disponibles

df = pd.read_csv('muestraTiempoHastaFallo.csv')
print(df)

x = df['x'].to_list()   #  <---------------------------------------------- trabaja con la variable "x"
n = len(x)

xs = np.linspace(min(x)-np.std(x),max(x)+np.std(x), 5000)
    
plt.figure(1, figsize=(15, 4))
plt.hist(x, bins=100, density = True, alpha = 0.4)
plt.title(f'Muestra con {n} datos');
plt.show()

### a) Para modelar la distribución del tiempo hasta el fallo ajusta a la muestra disponible una distribución log-normal, una triangular y una exponencial. Representa en la misma gráfica las funciones de densidad estimadas junto el histograma de la muestra. Elige la distribución que parece la más adecuada, justificando la respuesta (2 puntos)

In [ ]:
# Respuesta

# Lista de distribuciones continuas a probar
dist_names = ['lognorm', 'triang', 'expon']

xs = np.linspace(min(x)-np.std(x),max(x)+np.std(x), 5000)
    
plt.figure(1, figsize=(15, 6))
plt.hist(x, bins=40, density = True, alpha = 0.4)
    
# crea contendor con los resultados para elegir al final
res_aju_distr = []
    
for dist_name in dist_names:
    dist = getattr(stats, dist_name)
    params = dist.fit(x)
    arg = params[:-2]
    loc = params[-2]
    scale = params[-1]
    
    if arg:
        pdf_fitted = dist.pdf(xs, *arg, loc=loc, scale=scale)
        logLikelihood_med = np.sum(dist.logpdf(x, *arg, loc=loc, scale=scale)) / n
    else:
        pdf_fitted = dist.pdf(xs, loc=loc, scale=scale)
        logLikelihood_med = np.sum(dist.logpdf(x, loc=loc, scale=scale)) / n

    # guarda resultado para mostrarlo al final
    res_aju_distr.append((dist_name, logLikelihood_med, params))
        
    print(f'{dist_name} :: \t logLikelihood_med = {logLikelihood_med} \t params:{params}')
    plt.plot(xs, pdf_fitted, linewidth = 3, label=f'logLikelihood_med({dist_name}) = {logLikelihood_med}' )
    
plt.legend(loc='upper left')
plt.show()

# tabla de resultados, ordenada de peor a mejor según logLikelihood_med
res_aju_distr_ord = sorted(res_aju_distr, key=lambda x:x[1])
res_df = pd.DataFrame(res_aju_distr_ord, columns = ['Distribución', 'Media verosimilitud log', 'Parámetros'])
print(res_df)

# selección
print('\nLa distribución triangular queda rápidamente descartada ya que es la que hace menos verosimil la muestra (menor loglikelihood_med) '
      'además de estar claramente muy mal ajustada a lo observado en el histograma.')
print('La mejor opción en cuanto a error es la distribución log-normal ya que su ajuste es un poco mejor que la exponencial, que tiene menor loglikelihood_med.')
print('Sin embargo, la exponencial, con un error un poco mayor (menor VSL media), tiene un parámetro menos que ajustar (menos compleja). '
      'Tanto la log-normal como la exponencial son soluciones muy competitivas.')

### b) Como alternativa a la distribución paramétrica elegida en el apartado anterior, utiliza un estimador Kernel a partir de la muestra disponible. Como función kernel utiliza una gausiana y el ancho de banda estímalo automáticamente. Representa en la misma gráfica la función estimada con KDE y el histograma de la muestra. ¿Qué modelo tiene sentido elegir? Justifica la respuesta (2 puntos)

In [ ]:
# Respuesta

# fija la función kernel a utilizar
kernel_fun = 'gau'

# fija el bandwidth (si auto lo estima la función de ajuste)
tipo_bw = 'auto' # 'manual'

if tipo_bw == 'auto':
    bandwidth = 'normal_reference' # otras opciones: bandwidth = 'scott', bandwidth = 'silverman'
else:
    bandwidth = 0.6 # fija bandwidth exacto

# estima kde
kde = KDEUnivariate(x)
kde.fit(kernel = kernel_fun, bw = bandwidth, fft = True)

# calcula verosimilitud log media de ambas aproximaciones
logLikelihood_med_kde = np.sum(np.log(kde.evaluate(x))) / n

print(f'Verosimilitud logarítmica media de KDE: {logLikelihood_med_kde}')

# fija nbins para mostrar histograma
nbins_hist = 40

plt.figure(1, figsize=(15, 5))
plt.hist(x, nbins_hist, density = True, alpha = 0.4, label =f'Histograma nbins={nbins_hist}')
plt.plot(kde.support, kde.density, color = 'black', linewidth = 3, label =f'pdf KDE bw={kde.bw}')

# marca el máximo
index_max_pdf = np.argmax(kde.density)   # <--- encuentra el argumento que hace máximo la función
x_max_pdf = kde.support[index_max_pdf]
max_pdf = kde.density[index_max_pdf]
plt.plot(x_max_pdf, max_pdf, 's', color = 'green', markersize = 10, label =f'max pdf KDE ({x_max_pdf}, {max_pdf})')

sns.rugplot(x, height = 0.025, lw = 0.4, color = 'red')
plt.title(f'KDE con bandwidth = {kde.bw} | bw fijado con método: {kde.bw_method}')
plt.grid(); plt.legend()
plt.show()

# conclusión
print('Tiene más sentido utilizar la distribución log-normal o la exponencial como modelo del tiempo hasta el fallo ya que la pdf del KDE tiene problemas para modelar '
      'correctamente ambas colas de la distribución, siendo muy llamativo lo que ocurre en la cola de la izquierda (asigna densidades no nulas a valores muy pequeños), muy poco realista.')
print('En el caso de la log-normal, se consigue además un ajuste mejor, al ser la verosimilitud logarítmica media del KDE un poco menor que la conseguida con '
      'con la distribución log-normal.')

### c) A partir de la muestra estima la duración del periodo de garantía, medido en meses, que debería indicarse en el certificado de garantía del producto para que el mismo solo cubra el 75% de los fallos. Justifica la respuesta (1 punto)

In [ ]:
# Respuesta

print('Calculando directamente el percentil 75 de la muestra de tiempos hasta el fallo es posible estimar la duración solicitada.')

x_prctle = np.percentile(x, 75)
print(f'Por tanto, la duración del periodo de garantía debería ser de {x_prctle} meses.')
print(f'Que, por temas estéticos, se podría redondear a {round(x_prctle,0)} meses.')

print('\nOtra posible forma de obtener una estimación razonable es utilizar la función cuantil de la distribución log-normal '
      'estimada en el primer apartado.')

params_lognormal = res_aju_distr_ord[2][2]
arg = params_lognormal[:-2]
loc = params_lognormal[-2]
scale = params_lognormal[-1]
VA = stats.lognorm(s=arg, loc=loc, scale=scale)
x_q75 = VA.ppf(0.75)
print(f'Obteníendose un valor muy similar para la duración del periodo de garantía: {x_q75[0]} meses.')

params_exp = res_aju_distr_ord[1][2]
arg = params_exp[:-2]
loc = params_exp[-2]
scale = params_exp[-1]
VA = stats.expon(loc=loc, scale=scale)
x_q75 = VA.ppf(0.75)
print(f'\nSi se utiliza la distribución exponencial, entonces la duración del periodo de garantía es: {x_q75} meses.')
print('Queda claro que todo apunta a los 4 meses como un valor muy razonable.')

### Dentro de la empresa hay un grupo de ingenieros que afirma que el problema con estos cables no está en el diseño de los mismos, sino en la calidad de los materiales utilizados en la actualidad, que ha disminuido. En la empresa se dispone del tiempo medio hasta el fallo, determinado con precisión durante las exhaustivas pruebas de resistencia realizadas hace unos años, fijado finalmente en 5 meses. 

### d) Ayuda al grupo de ingenieros realizando un contraste de hipótesis paramétrico para el tiempo medio hasta el fallo. Plantea el contraste lógico a realizar, indicando las hipótesis H0 y H1. Resuelve el contraste de forma detallada a partir de la muestra disponible, con un nivel de significación del 1%. Calcula el valor del estadístico de contraste en la muestra, los valores críticos y el p-valor asociado. Razona utilizando por separado los valores críticos y el p-valor para tener más seguridad en lo que haces y completa con una representación gráfica de toda la información. No olvides concluir claramente. No olvides concluir claramente. (4 puntos)

In [ ]:
# Respuesta

mu_0 = 5

# datos de la muestra
n = len(x)
x_media = np.mean(x)
sx = np.std(x, ddof=1) # dividiendo por n-1
print(f'La media de la muestra es {x_media} y su desv. típica {sx}')

# Contraste de hipótesis para la media de X
print('\nEl contraste lógico a realizar es unilateral ("menor que"), ya que nos situamos en la posición aceptada en la actualidad '
     'en la empresa con la idea de poderla rechazar a favor de lo que están diciendo que está pasando los ingenieros:')
print(f' H0: mu_X = {mu_0}')
print(f' H1: mu_X < {mu_0}')

t0 = (x_media - mu_0) / (sx/np.sqrt(n))

# crea la distribución del estadístico de contraste
df = n -1 # grados de libertad
T0 = stats.t(df)

# calcula el pvalor (acorde a H1: mu < mu_0)
pvalor = T0.cdf(t0)

# determina los valores críticos que definen la región de rechazo (solo hay una situada a la izquierda)
alfa = 0.01  # nivel de significación
t_crit_iz = - T0.ppf(1-alfa)


print(f'El valor observado del estadistico de contraste T0 es {t0}, un valor pequeño considerando '
      f'que el error estandar de T0 es {(sx/np.sqrt(n))}')

print(f'La región de rechazo está situada a la izquierda, por lo que solo hay un valor crítico, que al 1% vale {t_crit_iz}')
print(f'Como t0 está dentro de la región de rechazo ({t0} < {t_crit_iz}), se puede rechazar la hipótesis nula al 1%.')

print(f'El p-valor asociado es igual a {pvalor}.')
print(f'Como el pvalor es menor que alfa ({pvalor} < {alfa}), hay que recharzar HO al 1% de significación, en favor de H1.')

print('Por tanto, el test de hipótesis ha resultado significativo. Se puede concluir al 1% de nivel de significación que el tiempo medio '
      'hasta el fallo en la actualidad es menor a los 5 meses establecidos hace años como una verdad absoluta en la empresa.')

In [ ]:
# Gráficamente para terminar de confirmar que todo está bien

print('Representación gráfica del contraste:')
# puntos para representar la distribución y las regiones de aceptación y rechazo
xmin = T0.ppf(0.00000000000000000001)
xmax = T0.ppf(1-0.000001)
xs = np.linspace(xmin, xmax, 1000)
xRA = np.linspace(t_crit_iz, xmax, 1000)
xRRiz = np.linspace(xmin, t_crit_iz, 1000)

plt.figure(1, figsize=(20, 6))
plt.plot(xs, T0.pdf(xs), linewidth = 4, label=f'Distribución T0 si H0 es cierta')

plt.fill_between( xRA, T0.pdf(xRA), color="green", alpha=0.6, label=f'Probabilidad asociada RA = {1-alfa}')
plt.fill_between( xRRiz, T0.pdf(xRRiz), color="red", alpha=0.6, label=f'P(T0>{t_crit_iz}) = {alfa}')

plt.vlines(t0, 0, np.max(T0.pdf(xs)), color='black', label=f't0 = {t0}')

plt.hlines(-0.02, t_crit_iz, xmax, color='green', label = f'RA: [{t_crit_iz}, inf] al {alfa*100}%')
plt.hlines(-0.02, xmin, t_crit_iz, color='red', linewidth = 6, label = f'RR')

plt.title('Distribución del estimador de contraste')
plt.xlabel('T0')
plt.legend()
plt.show()

print('Es evidente que hay que rechazar H0 en favor de H1, es decir, que con un nivel de confianza del 99% se puede afirmar que '
      'en la actualidad el tiempo medio hasta el fallo es menor que 5 meses. La tesis del grupo de ingenieros tiene pinta de ser correcta.')

In [ ]:
# Comprobación directa de los cálculos

tipo_alternativa =  'less' # alternativas posibles: ‘two-sided’, ‘less’, ‘greater’

# calculamos el estadístico en la muestra t0 y su pvalor
res = stats.ttest_1samp(x, popmean = mu_0, alternative = tipo_alternativa) 
print('Resultado: ', res)

t0 = res.statistic
pvalor = res.pvalue

print(f'El valor observado del estadistico de contraste es {t0}, con un p-valor asociado igual a {pvalor}.')
print('Los resultados son iguales a los calculados paso a paso en el apartado anterior, confirmando que hemos operado correctamente.')

### e) Estima un intervalo de confianza al 95% para mediana del tiempo hasta el fallo, utilizando bootstrap con 10000 submuestras y random_state = 999. Indica si el intervalo es simétrico, justificando la respuesta (1 punto)

In [ ]:
# Respuesta

x_mediana = np.median(x)

# params para llamar a bootstrap (método BCa, el que se suele usar normalmente)
muestra_orig = (x,)  # tiene que ser una secuencia
R = 10000
estadistico = np.median
nivel_confianza = 0.95

res_BCa = stats.bootstrap(muestra_orig, estadistico, confidence_level = nivel_confianza, 
                          n_resamples = R, method = 'BCA', random_state = 999)

# devuelve como resultado el IC y el SE del estadístico
lim_inf, lim_sup = res_BCa.confidence_interval
SE_BCa = res_BCa.standard_error

print(f'IC {100*nivel_confianza}% estimado según método BCa para la media: [{lim_inf}, {lim_sup}]')
print(f'Error estándar del estimador media muestral, estimado según método BCa: {SE_BCa}')

radio_iz = x_mediana - lim_inf
radio_de = lim_sup - x_mediana

print('\nComo se puede observar, el intervalo es muy asimétrico ya que el radio respecto a la mediana de la izquierda y de la derecha no son iguales:')
print(f'Radio_iz = {radio_iz}, Radio_de = {radio_de}. Es lógico que sea así debido a la asimetría de los datos de la muestra, con muy pocos datos a la '
      'derecha, que tendrán poca probabilidad de aparecer en las submuestras bootstrap.')

# graficamente
fig = plt.figure(1, figsize=(16, 1))
plt.errorbar(x_mediana, 1, xerr = np.array([[x_mediana - lim_inf], 
            [lim_sup - x_mediana]]), fmt = 'o', label = f'IC [{lim_inf}, {lim_sup}] al {nivel_confianza*100}%')

plt.legend()
plt.show()